# Plan-graph Vendi with learned GIN embeddings (cosine kernel)

Robustness check for the headline **plan-graph Vendi** in [`tables_2_3_4.ipynb`](tables_2_3_4.ipynb) (Table 2 there).
The published score uses a hand-built kernel (weighted Jaccard over operator / edge / 3-gram counts). Here the
kernel is replaced with **cosine similarity between learned plan embeddings**, and everything else stays the same:

| | Published Vendi | This notebook |
|---|---|---|
| Plans | `Workloads/<w>/plans.json` (compiled EXPLAIN DAGs, `status == ok`) | **same files**: no regeneration, no re-execution |
| Plan representation | operator / edge / n-gram feature counter | `TrinoGraphGINPlanEncoder` embedding (256-d) |
| Kernel | weighted Jaccard | cosine |
| Score | `vendi_score_from_similarity` | **same function** |

**Encoder setup = the QPP evaluation's.** The kwargs come straight from
`qpp_harness.gin_encoder_default_kwargs(SEED)` (hidden 128, 3 GINE layers, emb_dim 256, 200 epochs, mask 0.15,
closed-world `fixed_vocab`). They are used through the same `fit_from_train_plans` / `encode_plan` path, and
embeddings stay **unscaled** like `make_plan_embeddings_by_run`'s output. The one deliberate difference is the
training set: the encoder is trained **once, on the pooled plans of all 8 workloads**, and that one frozen
encoder embeds every workload. All workloads therefore share a single embedding space, which no per-workload
encoder could guarantee.

Rows follow Table 5's order (reference, baselines A–Z, ours last). The reference row is `tpcds_1200`, the
same directory `tables_2_3_4.ipynb` reads.

**Vocabulary caveat (§2).** The QPP `fixed_vocab` lacks three operators that occur in these plans, so they
fold into the shared `UNK` row. The headline run keeps the QPP setup verbatim. A second run swaps in a
vocabulary taken from the pooled plans (`fixed_vocab=None`) as a sensitivity check, and nothing else changes.

Training takes about 2 h per encoder on CPU. Encoders and embeddings are cached under `results/gin_vendi/`, so
re-running the notebook only reloads them (set `FORCE_RETRAIN = True` to refit).

## 0 · Setup

In [1]:
import sys, json, hashlib, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

ROOT = Path("/mnt/primary/Main")
for _p in (str(ROOT), str(ROOT / "workload_generation" / "qpp_evaluation")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from qpp_harness import gin_encoder_default_kwargs          # the QPP evaluation's GIN config
from uncertainty_prediction.src import TrinoGraphGINPlanEncoder
from uncertainty_prediction.new_config import SEED           # same base seed as the QPP notebooks
from loader.workload_analysis import dags_from_plan_bundle, vendi_score_from_similarity

WORKLOAD_ROOT = ROOT / "Workloads"
RESULTS_DIR = Path("./results").resolve()
CACHE_DIR = RESULTS_DIR / "gin_vendi"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RETRAIN = False

# Table 2's 8 workloads, in Table 5's row order: reference first, baselines (A-Z), ours last.
REF       = "tpcds_1200"   # the reference dir tables_2_3_4.ipynb reads (1208 plans)
OURS      = ["gpt_generated_1000_tpcds_tokens_no_feedback", "gpt_generated_1000_tpcds_tokens"]
BASELINES = ["bootstrapping_lcm_1000_tokens_tpcds",
             "e2etune_1000_tokens_tpcds",
             "sql_factory_1000_tokens_tpcds",
             "sqlbarber_1000_tokens_tpcds",
             "sqlstorm_1000_tokens_tpcds"]
ROWS = [REF] + BASELINES + OURS
HEADLINE = "gpt_generated_1000_tpcds_tokens"   # DiverSQL (W/ Feedback)

PRETTY = {"tpcds_1200":"TPC-DS (1000)", "gpt_generated_1000_tpcds_tokens":"DiverSQL (W/ Feedback)", "gpt_generated_1000_tpcds_tokens_no_feedback":"DiverSQL (W/O Feedback)",
          "sqlstorm_1000_tokens_tpcds":"SQLStorm", "sqlbarber_1000_tokens_tpcds":"SQLBarber",
          "sql_factory_1000_tokens_tpcds":"SQL-Factory", "e2etune_1000_tokens_tpcds":"E2ETune",
          "bootstrapping_lcm_1000_tokens_tpcds":"Bootstrapping-LCM"}
_INV = {v:k for k,v in PRETTY.items()}

def show(tbl, fmt="{:.3f}", **kw):
    disp = tbl.rename(index=PRETTY)
    def hl(row):
        key = _INV.get(row.name, row.name)
        if key == REF:  return ["background-color:#f3f4f6;font-style:italic"]*len(row)
        if key in OURS: return ["background-color:#fff1e6;font-weight:600"]*len(row)
        return [""]*len(row)
    sty = disp.style.apply(hl, axis=1)
    return sty.format(fmt, **kw) if fmt else sty


## 1 · Load the existing compiled plans

`plans.json` is the cached EXPLAIN bundle written by `analysis.capture_query_plans`. `dags_from_plan_bundle`
keeps only `status == ok` plans, the same filter the published Vendi used. Nothing here talks to Trino.

In [2]:
dags_by_workload = {}
plan_meta = []
for w in ROWS:
    bundle = json.loads((WORKLOAD_ROOT / w / "plans.json").read_text(encoding="utf-8"))
    dags_by_workload[w] = dags_from_plan_bundle(bundle)
    plan_meta.append({"workload": w, "plans_ok": len(dags_by_workload[w]),
                      "query_count": bundle.get("query_count"), "captured_at_utc": bundle.get("captured_at_utc")})

df_plans = pd.DataFrame(plan_meta).set_index("workload")
display(show(df_plans, None))

# pooled training set: every plan of every workload, in a fixed order
POOLED = [(w, q) for w in ROWS for q in dags_by_workload[w]]
POOLED_PLANS = [dags_by_workload[w][q] for w, q in POOLED]
print(f"pooled training plans: {len(POOLED_PLANS)}")


,plans_ok,query_count,captured_at_utc
workload,,,
TPC-DS (1000),1208,1208,2026-09-28T12:59:42.682105+00:00
Bootstrapping-LCM,1000,1000,2026-08-10T13:51:19.260689+00:00
E2ETune,1000,1000,2026-08-10T13:55:07.200383+00:00
SQL-Factory,1000,1000,2026-08-10T13:58:34.956926+00:00
SQLBarber,1000,1000,2026-08-10T13:11:36.092271+00:00
SQLStorm,1000,1000,2026-08-10T14:04:44.942261+00:00
DiverSQL (W/O Feedback),1000,1000,2026-08-10T12:37:04.981274+00:00
DiverSQL (W/ Feedback),1000,1000,2026-08-10T12:30:00.664987+00:00


pooled training plans: 8208


## 2 · Encoder config (QPP setup) and the vocabulary check

`fixed_vocab` is `expected_operator_types(read_only=True, iceberg=True)`, the schema's reachable-operator set.
Any operator outside it maps to `UNK`. The cell below lists which observed operators fall outside it and which
workloads contain them, since collapsing them into one row could understate some workloads' diversity.

In [8]:
QPP_KWARGS = gin_encoder_default_kwargs(seed=SEED)

_vocab = set(QPP_KWARGS["fixed_vocab"])
_oov = {w: Counter(n["name"] for d in dags_by_workload[w].values() for n in d["nodes"].values()
                   if n["name"] not in _vocab) for w in ROWS}
oov_tbl = pd.DataFrame(_oov).T.fillna(0).astype(int).reindex(ROWS)
VARIANTS = {
    "qpp":    dict(QPP_KWARGS),
    "pooled": {**QPP_KWARGS, "fixed_vocab": None},
}


## 3 · Train once on the pooled plans, embed every workload

One `fit_from_train_plans` per variant on all 8,208 pooled plans (seed = `SEED`), then a frozen `encode_plan`
for every plan. Results are cached under `results/gin_vendi/` together with a fingerprint of the kwargs and the
plan set. The cache is only reused when both still match, so stale embeddings can't be picked up silently.

In [4]:
def _fingerprint(kwargs):
    kw = {k: (sorted(v) if isinstance(v, (set, frozenset)) else v)
          for k, v in kwargs.items() if k not in ("device", "verbose")}
    plans_sig = [(w, q, len(d["nodes"]), len(d["edges"])) for (w, q), d in zip(POOLED, POOLED_PLANS)]
    blob = json.dumps({"kwargs": kw, "plans": plans_sig}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def fit_or_load(name, kwargs):
    fp = _fingerprint(kwargs)
    enc_path, emb_path = CACHE_DIR / f"gin_{name}_seed{SEED}.pt", CACHE_DIR / f"emb_{name}_seed{SEED}.npz"
    if not FORCE_RETRAIN and emb_path.exists():
        z = np.load(emb_path, allow_pickle=False)
        if str(z["fingerprint"]) == fp:
            print(f"[{name}] loaded cached embeddings ({fp})")
            return z["Z"]
        print(f"[{name}] cache fingerprint mismatch -> retraining")

    t0 = time.time()
    enc = TrinoGraphGINPlanEncoder.fit_from_train_plans(POOLED_PLANS, **{**kwargs, "verbose": True})
    print(f"[{name}] trained in {(time.time() - t0) / 60:.1f} min, dim={enc.dim()}, vocab={len(enc._op2idx)}")
    Z = enc.encode_plans(POOLED_PLANS)          # frozen, unscaled -- as make_plan_embeddings_by_run returns
    enc.save(str(enc_path))
    np.savez(emb_path, Z=Z, fingerprint=fp,
             workload=np.array([w for w, _ in POOLED]), query=np.array([q for _, q in POOLED]))
    return Z

Z_BY_VARIANT = {name: fit_or_load(name, kw) for name, kw in VARIANTS.items()}


[GIN pretext] epoch    0  loss=4.9372
[GIN pretext] epoch   20  loss=1.3643
[GIN pretext] epoch   40  loss=1.3178
[GIN pretext] epoch   60  loss=1.3007
[GIN pretext] epoch   80  loss=1.2864
[GIN pretext] epoch  100  loss=1.2841
[GIN pretext] epoch  120  loss=1.2739
[GIN pretext] epoch  140  loss=1.2749
[GIN pretext] epoch  160  loss=1.2712
[GIN pretext] epoch  180  loss=1.2687
[GIN pretext] epoch  199  loss=1.2648
[qpp] trained in 2.7 min, dim=256, vocab=54
[GIN pretext] epoch    0  loss=4.9203
[GIN pretext] epoch   20  loss=0.4925
[GIN pretext] epoch   40  loss=0.4298
[GIN pretext] epoch   60  loss=0.3978
[GIN pretext] epoch   80  loss=0.3798
[GIN pretext] epoch  100  loss=0.3700
[GIN pretext] epoch  120  loss=0.3524
[GIN pretext] epoch  140  loss=0.3496
[GIN pretext] epoch  160  loss=0.3486
[GIN pretext] epoch  180  loss=0.3399
[GIN pretext] epoch  199  loss=0.3360
[pooled] trained in 2.6 min, dim=256, vocab=38


## 4 · Plan-graph Vendi with a cosine kernel

For each workload, `K = Ẑ Ẑᵀ` with `Ẑ` the L2-normalised embeddings (so `K_ii = 1` and K is PSD). `K` then goes
through the **same** `vendi_score_from_similarity` as the published metric. The published Jaccard-kernel
Vendi is read from the `results/*.csv` files exactly as `tables_2_3_4.ipynb` reads them.

Two embedding readings per encoder:
- **raw** (headline): `encode_plan` output exactly as the QPP pipeline hands it downstream.
- **standardised**: a `StandardScaler` fitted on the pooled embeddings and applied before cosine. This is the
  per-dimension standardisation the QPP regressors apply to the same embeddings (`use_scaler=True`). It is
  fitted once on the pooled set, so every workload is still measured in one shared space. Sum-pooled GIN
  readouts share a large common direction, which can push every raw cosine towards 1. Centring removes that
  shared offset, and the mean-cosine diagnostic below shows whether it matters.

In [5]:
from sklearn.preprocessing import StandardScaler

def _unit(Z):
    norms = np.linalg.norm(Z, axis=1, keepdims=True)
    assert (norms > 0).all(), "zero-norm embedding: cosine undefined"
    return Z / norms

def cosine_vendi(Z):
    Zn = _unit(Z)
    return vendi_score_from_similarity(Zn @ Zn.T)

def mean_offdiag_cosine(Z):
    Zn = _unit(Z); K = Zn @ Zn.T; n = len(K)
    return float((K.sum() - np.trace(K)) / (n * (n - 1)))

_wl = np.array([w for w, _ in POOLED])
EMB = {}   # column label -> pooled embedding matrix
for name, Z in Z_BY_VARIANT.items():
    vocab = "QPP vocab" if name == "qpp" else "pooled vocab"
    EMB[f"GIN cosine, {vocab}, raw"] = Z
    EMB[f"GIN cosine, {vocab}, std."] = StandardScaler().fit_transform(Z)
HEADLINE_COL = "GIN cosine, QPP vocab, raw"

# published (weighted-Jaccard) Vendi -- same CSV merge as tables_2_3_4.ipynb
_df = None
for _p in sorted(RESULTS_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime):
    _d = pd.read_csv(_p)
    if "workload" not in _d.columns:
        continue
    _d = _d.set_index("workload")
    _df = _d if _df is None else _d.combine_first(_df)

table = pd.DataFrame({"Plans": df_plans["plans_ok"],
                      "Jaccard (published)": _df.loc[ROWS, "plan_graph_vendi_score"].astype(float)}, index=ROWS)
for col, Z in EMB.items():
    table[col] = [cosine_vendi(Z[_wl == w]) for w in ROWS]
VENDI_COLS = [c for c in table.columns if c != "Plans"]

# ranks among the 7 generators (reference excluded, as in the paper's best-value highlighting)
competitors = [w for w in ROWS if w != REF]
ranks = table.loc[competitors, VENDI_COLS].rank(ascending=False, method="min").reindex(ROWS)

# diagnostic: how collapsed is each cosine space? (1.0 = every plan points the same way)
mean_cos = pd.DataFrame({col: [mean_offdiag_cosine(Z[_wl == w]) for w in ROWS] for col, Z in EMB.items()}, index=ROWS)

print("Plan-graph Vendi (Table 5 row order)")
display(show(table, None).format({"Plans": "{:.0f}", **{c: "{:.2f}" for c in VENDI_COLS}}))
print("Rank among the 7 generators (1 = most diverse)")
display(show(ranks, "{:.0f}", na_rep="—"))
print("Mean off-diagonal cosine similarity (diagnostic)")
display(show(mean_cos, "{:.3f}"))


Plan-graph Vendi (Table 5 row order)


,Plans,Jaccard (published),"GIN cosine, QPP vocab, raw","GIN cosine, QPP vocab, std.","GIN cosine, pooled vocab, raw","GIN cosine, pooled vocab, std."
TPC-DS (1000),1208,22.70,1.01,2.16,2.66,10.53
Bootstrapping-LCM,1000,18.98,1.01,3.25,2.28,7.91
E2ETune,1000,55.32,1.04,2.27,3.48,15.87
SQL-Factory,1000,52.44,1.02,1.89,3.29,12.68
SQLBarber,1000,16.38,1.09,1.10,2.73,5.02
SQLStorm,1000,77.64,1.01,2.20,3.09,11.84
DiverSQL (W/O Feedback),1000,87.10,1.02,2.68,3.19,20.71
DiverSQL (W/ Feedback),1000,106.06,1.03,2.34,3.09,18.10


Rank among the 7 generators (1 = most diverse)


,Jaccard (published),"GIN cosine, QPP vocab, raw","GIN cosine, QPP vocab, std.","GIN cosine, pooled vocab, raw","GIN cosine, pooled vocab, std."
TPC-DS (1000),—,—,—,—,—
Bootstrapping-LCM,6,7,1,7,6
E2ETune,4,2,4,1,3
SQL-Factory,5,4,6,2,4
SQLBarber,7,1,7,6,7
SQLStorm,3,6,5,5,5
DiverSQL (W/O Feedback),2,5,2,3,1
DiverSQL (W/ Feedback),1,3,3,4,2


Mean off-diagonal cosine similarity (diagnostic)


,"GIN cosine, QPP vocab, raw","GIN cosine, QPP vocab, std.","GIN cosine, pooled vocab, raw","GIN cosine, pooled vocab, std."
TPC-DS (1000),0.998,0.047,0.764,0.226
Bootstrapping-LCM,0.998,0.363,0.825,0.436
E2ETune,0.993,0.513,0.680,0.240
SQL-Factory,0.997,0.681,0.706,0.306
SQLBarber,0.984,0.984,0.749,0.619
SQLStorm,0.998,0.033,0.704,0.103
DiverSQL (W/O Feedback),0.997,0.011,0.724,0.030
DiverSQL (W/ Feedback),0.996,0.230,0.723,0.084


In [6]:
table.join(ranks.add_prefix("Rank: ")).to_csv(CACHE_DIR / "plan_vendi_gin_cosine.csv")
mean_cos.to_csv(CACHE_DIR / "plan_vendi_gin_cosine_mean_cos.csv")

def verdict(col):
    s = table.loc[competitors, col].sort_values(ascending=False)
    top, runner = s.index[0], s.index[1]
    first = top == HEADLINE
    lead = s.iloc[0] - s.iloc[1]
    return (f"{col:32s} {'YES' if first else 'NO ':3s}  first: {PRETTY[top]} ({s.iloc[0]:.2f}), "
            f"runner-up: {PRETTY[runner]} ({s.iloc[1]:.2f}), lead {lead:+.2f};  "
            f"DiverSQL (W/ Feedback) rank {int(ranks.loc[HEADLINE, col])}")

print("Does DiverSQL (W/ Feedback) still rank first?")
for col in VENDI_COLS:
    print(" ", verdict(col))

print("\nSpearman rank correlation with the published Jaccard Vendi (7 generators):")
for col in EMB:
    rho = table.loc[competitors, [col, "Jaccard (published)"]].corr(method="spearman").iloc[0, 1]
    print(f"  {col:32s} rho = {rho:+.3f}")


Does DiverSQL (W/ Feedback) still rank first?
  Jaccard (published)              YES  first: DiverSQL (W/ Feedback) (106.06), runner-up: DiverSQL (W/O Feedback) (87.10), lead +18.96;  DiverSQL (W/ Feedback) rank 1
  GIN cosine, QPP vocab, raw       NO   first: SQLBarber (1.09), runner-up: E2ETune (1.04), lead +0.05;  DiverSQL (W/ Feedback) rank 3
  GIN cosine, QPP vocab, std.      NO   first: Bootstrapping-LCM (3.25), runner-up: DiverSQL (W/O Feedback) (2.68), lead +0.57;  DiverSQL (W/ Feedback) rank 3
  GIN cosine, pooled vocab, raw    NO   first: E2ETune (3.48), runner-up: SQL-Factory (3.29), lead +0.19;  DiverSQL (W/ Feedback) rank 4
  GIN cosine, pooled vocab, std.   NO   first: DiverSQL (W/O Feedback) (20.71), runner-up: DiverSQL (W/ Feedback) (18.10), lead +2.61;  DiverSQL (W/ Feedback) rank 2

Spearman rank correlation with the published Jaccard Vendi (7 generators):
  GIN cosine, QPP vocab, raw       rho = -0.143
  GIN cosine, QPP vocab, std.      rho = +0.393
  GIN cosine, poo

## Notes

- **One encoder fit per variant, one seed (`SEED`).** The QPP sweeps refit per seed. This check deliberately
  trains once, as specified, so seed-to-seed variance of the GIN embedding space is not measured here.
- **Embeddings are unscaled** (raw `encode_plan` output, matching what `make_plan_embeddings_by_run` hands
  downstream). The QPP regressors standardise internally, but that happens downstream of the encoder and is
  not part of the embedding.
- **Vendi is not size-normalised.** The reference row has 1,208 plans against 1,000 for every generator, as in
  the published table. The reference is context, not a competitor, and is excluded from the ranks.
- Absolute scales differ between kernels (a learned cosine space is denser than sparse Jaccard features), so
  compare **ranks and ordering** across columns, not raw magnitudes.